# FunSearch — String Compression

Evolving a `compress(s) → str` function. The LLM can't one-shot this because the scoring depends on **our specific test strings** — no textbook answer.

**Score** = average compression ratio across test strings. Must decompress back to original (correctness check). Higher ratio = better.

## 1. LLM Setup

In [15]:
import os
from litellm import completion
import litellm
from statistics import mean
from dataclasses import dataclass

model_name = "nvidia_nim/openai/gpt-oss-20b"
litellm.register_model({model_name: {
    "max_tokens": 8192,
    "input_cost_per_token": 0.0,
    "output_cost_per_token": 0.0,
    "supports_assistant_prefill": False,
}})

{'nvidia_nim/openai/gpt-oss-20b': {'max_tokens': 8192,
  'input_cost_per_token': 0.0,
  'output_cost_per_token': 0.0,
  'supports_assistant_prefill': False}}

## 2. Sampler

In [16]:
def sample_code(prompt: str) -> str:
    resp = completion(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
    )
    text = resp.choices[0].message.content
    if "```" in text:
        code = text.split("```")[1]
        if code.startswith("python"):
            code = code[len("python"):]
        return code.strip()
    return text.strip()

## 3. Programs Database

In [17]:
@dataclass
class DB:
    code : str
    score : float

    def __str__(self):
        preview = self.code.strip().split('\n')[0]
        return f"[score={self.score:.4f}] {preview}"

lis = []

def db_add(code, score):
    lis.append(DB(code=code, score=score))
    lis.sort(key=lambda x: x.score, reverse=True)

def best(k): return lis[:k]

## 4. Prompt Builder

In [18]:
def build_prompt(spec, best_programs):
    examples = "\n\n".join(
        f"# score: {p.score}\n{p.code}" for p in best_programs
    )
    return f"""Here's the task: {spec}

Here are the best solutions so far:
{examples}

Write an improved version. Return ONLY the function, no explanation."""

## 5. Compression Evaluator
Score = average compression ratio. The function must output `(compress, decompress)` — two functions. Correctness: `decompress(compress(s)) == s`. Shorter output = higher score.

In [19]:
test_strings = [
    "aaaaaabbbbbccccdddeef",
    "the cat sat on the mat and the cat saw the bat",
    "abcabcabcabcabcabc",
    "aaaaaaaaaaaaaaaaaaaabbbbbbbbbbbbbbbbcccccccccc",
    "xyxyxyxyxyxyxyxyxyxy",
    "hello world hello world hello world",
    "abcdefghijklmnopqrstuvwxyz",
    "aaabbbcccaaabbbcccaaabbbccc",
]

def evaluate_compression(code_str):
    """Score = avg compression ratio. 0 if decompress fails or output is longer."""
    namespace = {}
    try:
        exec(code_str, namespace)
    except Exception:
        return 0.0

    if "compress" not in namespace or "decompress" not in namespace:
        return 0.0

    ratios = []
    for s in test_strings:
        try:
            compressed = namespace["compress"](s)
            decompressed = namespace["decompress"](compressed)
            if decompressed != s:
                ratios.append(0.0)
            elif len(compressed) >= len(s):
                ratios.append(0.0)
            else:
                ratios.append(1.0 - len(compressed) / len(s))
        except Exception:
            ratios.append(0.0)

    return sum(ratios) / len(ratios)

## 6. Seed Program + Spec

In [20]:
spec = """Write two functions: `compress(s)` and `decompress(s)`.
- compress takes a string and returns a shorter string encoding
- decompress takes the compressed string and returns the original
- decompress(compress(s)) must equal s exactly
- goal: make the compressed output as short as possible
- do NOT use any imports or standard library compression"""

seed = '''def compress(s):
    out = []
    i = 0
    while i < len(s):
        c = s[i]
        count = 1
        while i + count < len(s) and s[i + count] == c:
            count += 1
        if count > 1:
            out.append(f"{count}{c}")
        else:
            out.append(c)
        i += count
    return "".join(out)

def decompress(s):
    out = []
    i = 0
    while i < len(s):
        if s[i].isdigit():
            num = []
            while i < len(s) and s[i].isdigit():
                num.append(s[i])
                i += 1
            out.append(int("".join(num)) * s[i])
            i += 1
        else:
            out.append(s[i])
            i += 1
    return "".join(out)'''

lis = []
seed_score = evaluate_compression(seed)
db_add(seed, seed_score)
print(f"Seed score: {seed_score:.4f}")

Seed score: 0.2017


## 7. The Loop — evolve the packing strategy

In [21]:
for i in range(10):
    prompt = build_prompt(spec, best(2))
    new_code = sample_code(prompt)
    score = evaluate_compression(new_code)
    db_add(new_code, score)

    if score == 0.0:
        # debug: why did it fail?
        ns = {}
        try:
            exec(new_code, ns)
            if "compress" in ns and "decompress" in ns:
                t = test_strings[0]
                c = ns["compress"](t)
                d = ns["decompress"](c)
                print(f"  iter {i}: score=0 | orig={t!r} comp={c!r} decomp={d!r} match={d==t}")
            else:
                print(f"  iter {i}: score=0 | missing compress/decompress")
        except Exception as e:
            print(f"  iter {i}: score=0 | exec error: {e}")
    else:
        print(f"  iter {i}: score={score:.4f}  best={best(1)[0].score:.4f}")

print(f"\n--- Best solution ---")
winner = best(1)[0]
print(winner)
print(winner.code)

  iter 0: score=0.1363  best=0.2017
  iter 1: score=0.2485  best=0.2485
  iter 2: score=0.4437  best=0.4437
  iter 3: score=0.2287  best=0.4437
  iter 4: score=0.1471  best=0.4437
  iter 5: score=0.3006  best=0.4437
  iter 6: score=0 | orig='aaaaaabbbbbccccdddeef' comp='aaaaaabbbbbccccdddeef' decomp='aaaaaabbbbbccccdddeef' match=True
  iter 7: score=0.4437  best=0.4437
  iter 8: score=0.4437  best=0.4437
  iter 9: score=0.4383  best=0.4437

--- Best solution ---
[score=0.4437] def compress(s):
def compress(s):
    out = []
    i, n = 0, len(s)
    window_size = 4096
    while i < n:
        # longest match in previous window
        start = max(0, i - window_size)
        best_len, best_off = 0, 0
        for j in range(start, i):
            l = 0
            while l < 255 and i + l < n and s[j + l] == s[i + l]:
                l += 1
            if l > best_len:
                best_len, best_off = l, i - j
                if l == 255:
                    break
        if best_len >=

## 8. Experiment: No Seed — does the LLM do worse without a starting point?

In [22]:
lis = []  # no seed — start empty

for i in range(10):
    prompt = build_prompt(spec, best(2))
    new_code = sample_code(prompt)
    score = evaluate_compression(new_code)
    db_add(new_code, score)

    if score == 0.0:
        ns = {}
        try:
            exec(new_code, ns)
            if "compress" in ns and "decompress" in ns:
                t = test_strings[0]
                c = ns["compress"](t)
                d = ns["decompress"](c)
                print(f"  iter {i}: score=0 | comp={c!r} match={d==t}")
            else:
                print(f"  iter {i}: score=0 | missing funcs")
        except Exception as e:
            print(f"  iter {i}: score=0 | error: {e}")
    else:
        print(f"  iter {i}: score={score:.4f}  best={best(1)[0].score:.4f}")

print(f"\nNo-seed best: {best(1)[0].score:.4f}" if lis else "\nAll failed")

  iter 0: score=0.0842  best=0.0842
  iter 1: score=0.0924  best=0.0924
  iter 2: score=0.0924  best=0.0924
  iter 3: score=0.1102  best=0.1102
  iter 4: score=0.0924  best=0.1102
  iter 5: score=0.1102  best=0.1102
  iter 6: score=0.1102  best=0.1102
  iter 7: score=0.1043  best=0.1102
  iter 8: score=0.1102  best=0.1102
  iter 9: score=0.1102  best=0.1102

No-seed best: 0.1102
